# GNN Benchmark — Multi-Config Evaluation

Trains the EGNN on every `.yaml` found in `BENCHMARK_DIR`, then shows per-experiment
Stage 0 / Stage 1 / Stage 2 plots and a cross-experiment comparison.

**To run a different folder**: change `BENCHMARK_DIR` in the next cell and restart.

| Metric | Meaning |
|--------|---------|
| `chamfer` | Unweighted Chamfer distance (lower = tighter circle fit) |
| `energy` | Total physical energy at Stage 2 equilibrium |
| `hinge_gap` | Weighted hinge connectivity residual (lower = better connected) |
| `total_loss` | Full training objective at last epoch |

In [ ]:
import os, sys, glob, copy, time
os.environ["JAX_PLATFORM_NAME"] = "cpu"

_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.insert(0, _root)
sys.path.insert(0, os.path.join(_root, 'src'))

import jax
jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from types import SimpleNamespace

from topology.builder import build_tessellation
from problem.conditions import configure_tessellation
from problem.config import load_and_parse_config
from jax_backend.state import CentroidalState
from jax_backend.pipeline import forward_pipeline
from jax_backend.training.trainer import train_pipeline
from jax_backend.gnn.graph_builder import build_static_graph_features
from jax_backend.gnn.egnn import init_egnn
from jax_backend.geometry import reconstruct_vertices, compute_face_areas
from jax_backend.utils.linalg import rotation_matrix
from utils.visualization import plot_tessellation

print(f"JAX backend: {jax.default_backend()}")

In [ ]:
# ── CONFIGURE HERE ────────────────────────────────────────────────────────────
BENCHMARK_DIR = os.path.join(_root, "data/configs/gnn/benchmark")
# ─────────────────────────────────────────────────────────────────────────────

config_paths = sorted(glob.glob(os.path.join(BENCHMARK_DIR, "*.yaml")))
print(f"Found {len(config_paths)} configs in {BENCHMARK_DIR}:")
for p in config_paths:
    print(f"  {os.path.basename(p)}")

In [ ]:
# ── HELPER FUNCTIONS ──────────────────────────────────────────────────────────

def setup_experiment(config_path):
    """Load config, build and center tessellation, init EGNN params."""
    config = load_and_parse_config(config_path)
    topo = config.topology
    topo_obj = SimpleNamespace(**topo)

    tessellation = build_tessellation(topo.get('pattern'), topo.get('width', 5), topo.get('height', 5))

    requested_area = topo.get('total_area')
    if requested_area:
        scale = np.sqrt(requested_area / tessellation.compute_total_area())
        tessellation.update_vertices(tessellation.vertices * scale)

    configure_tessellation(tessellation, topo_obj)

    target_center = np.array(getattr(config.target, 'center', [0.0, 0.0]), dtype=float)
    tess_centroid = np.mean(tessellation.get_face_centroids(), axis=0)
    tessellation.update_vertices(tessellation.vertices - tess_centroid + target_center)

    initial_state = CentroidalState.from_tessellation(tessellation, target_cfg=config.target)

    static_features = build_static_graph_features(initial_state)
    node_feat_dim = static_features['node_feat_dim']
    gnn_cfg = config.mapping.params if isinstance(config.mapping.params, dict) else {}
    hidden_dim = int(gnn_cfg.get('hidden_dim', 64))
    num_layers = int(gnn_cfg.get('num_layers', 4))
    seed = int(gnn_cfg.get('seed', 2))
    key = jax.random.PRNGKey(seed)
    init_params = init_egnn(key, node_feat_dim, hidden_dim, num_layers)

    return config, tessellation, initial_state, init_params


def run_training(config, tessellation, initial_state, init_params):
    """Train EGNN and run final forward pass."""
    optimized_params, history = train_pipeline(
        init_params, initial_state,
        config.target, config.validity, config.physics, config.training,
        map_type=config.mapping.type,
        use_shirley_chiu=config.mapping.use_shirley_chiu,
        strict_boundary_fit=config.mapping.strict_boundary_fit,
        learn_global_scale=config.mapping.learn_global_scale,
        use_jit=False,
    )
    result = forward_pipeline(
        initial_state, config.target, config.validity, config.physics,
        map_type=config.mapping.type,
        map_params=optimized_params,
        use_shirley_chiu=config.mapping.use_shirley_chiu,
        strict_boundary_fit=config.mapping.strict_boundary_fit,
    )
    return result, history, optimized_params


def tess_from_state(state, tessellation):
    """Reconstruct tessellation vertices from a CentroidalState."""
    verts_rec = np.array(reconstruct_vertices(state.face_centroids, state.centroid_node_vectors))
    tess_copy = copy.deepcopy(tessellation)
    new_verts = np.zeros_like(tess_copy.vertices)
    for i, face in enumerate(tess_copy.faces):
        for j, v_idx in enumerate(face.vertex_indices):
            new_verts[v_idx] = verts_rec[i, j]
    tess_copy.update_vertices(new_verts)
    return tess_copy


def equilibrium_state(valid_state, solution):
    """Apply Stage 2 displacements to produce the equilibrium CentroidalState."""
    final_fields = solution.fields[-1]
    c_eq = valid_state.face_centroids + final_fields[:, :2]
    R = jax.vmap(rotation_matrix)(final_fields[:, 2])
    s_eq = jnp.einsum('nij, nkj -> nki', R, valid_state.centroid_node_vectors)
    return valid_state._replace(face_centroids=c_eq, centroid_node_vectors=s_eq)


def extract_metrics(history, result):
    """Pull final-epoch scalars from training history and physics solution."""
    final = history[-1]
    chamfer   = float(final.get('chamfer_total', float('nan')))
    hinge_gap = float(final.get('hinge_gap', 0.0))
    total     = float(final.get('total', float('nan')))
    sol = result['solution']
    energy    = float(sol.energies['stretch'][-1] + sol.energies['shear'][-1] + sol.energies['contact'][-1])
    stretch   = float(sol.energies['stretch'][-1])
    shear     = float(sol.energies['shear'][-1])
    contact   = float(sol.energies['contact'][-1])
    return dict(chamfer=chamfer, hinge_gap=hinge_gap, total=total,
                energy=energy, stretch=stretch, shear=shear, contact=contact)


def extract_series(history, key):
    return [float(h[key]) for h in history if key in h]


print("Helpers ready.")

## Train all experiments

Results are stored in `experiments` — a list of dicts with keys:
`name`, `config`, `result`, `history`, `tess`, `metrics`.

In [ ]:
experiments = []

for cfg_path in config_paths:
    name = os.path.splitext(os.path.basename(cfg_path))[0]
    sep = '=' * 62
    print(f"\n{sep}\n  {name}\n{sep}")
    t0 = time.time()

    cfg, tess, s0, params0 = setup_experiment(cfg_path)
    res, hist, opt_p = run_training(cfg, tess, s0, params0)
    metrics = extract_metrics(hist, res)

    experiments.append(dict(name=name, config=cfg, result=res,
                            history=hist, tess=tess, metrics=metrics))
    print(f"  chamfer={metrics['chamfer']:.4f}  energy={metrics['energy']:.4f}  "
          f"hinge_gap={metrics['hinge_gap']:.2f}  [{time.time()-t0:.0f}s]")

print(f"\nAll {len(experiments)} experiments done.")

## Per-experiment visualisation

Each block shows: **Stage 0** (GNN mapping) | **Stage 1** (geometric validity) | **Stage 2** (physical equilibrium) + loss curve.

In [ ]:
def plot_experiment(exp):
    name   = exp['name']
    cfg    = exp['config']
    res    = exp['result']
    hist   = exp['history']
    tess   = exp['tess']
    m      = exp['metrics']

    target_params = {'type': cfg.target.type,
                     'center': cfg.target.center,
                     'radius': cfg.target.radius}
    plot_kw = dict(show_target=True, target_params=target_params,
                   show_hinges=True, show_hinge_indices=False,
                   show_face_indices=False, show_external_forces=True,
                   show_kinematic_blocks=True)

    fig = plt.figure(figsize=(20, 5))
    gs  = gridspec.GridSpec(1, 4, figure=fig, wspace=0.3)
    axes_stages = [fig.add_subplot(gs[0, i]) for i in range(3)]
    ax_loss     = fig.add_subplot(gs[0, 3])

    fig.suptitle(
        f"{name}   |   chamfer={m['chamfer']:.4f}   energy={m['energy']:.3f}   hinge_gap={m['hinge_gap']:.1f}",
        fontsize=11, y=1.02
    )

    # Stage 0
    t0_tess = tess_from_state(res['mapped_state'], tess)
    plot_tessellation(t0_tess, ax=axes_stages[0], title="Stage 0: GNN mapping",
                      original_vertices=tess.vertices, **plot_kw)

    # Stage 1
    t1_tess = tess_from_state(res['valid_state'], tess)
    plot_tessellation(t1_tess, ax=axes_stages[1], title="Stage 1: Geometric validity",
                      original_vertices=tess.vertices, **plot_kw)

    # Stage 2
    eq_state = equilibrium_state(res['valid_state'], res['solution'])
    t2_tess  = tess_from_state(eq_state, tess)
    plot_tessellation(t2_tess, ax=axes_stages[2], title="Stage 2: Physical equilibrium",
                      original_vertices=tess.vertices, **plot_kw)

    # Loss curve
    total   = extract_series(hist, 'total')
    chamfer = extract_series(hist, 'chamfer_total')
    hinge   = extract_series(hist, 'hinge_gap')
    energy  = extract_series(hist, 'energy')
    epochs  = range(len(total))
    ax_loss.plot(epochs, total,   label='Total',      lw=1.5, color='black')
    ax_loss.plot(epochs, chamfer, label='Chamfer',    lw=1.2, linestyle='--', color='#E63946')
    ax_loss.plot(epochs, hinge,   label='Hinge gap',  lw=1.2, linestyle=':',  color='#457B9D')
    if any(e > 1e-10 for e in energy):
        ax_loss.plot(epochs, energy, label='Energy', lw=1.0, linestyle='-.', color='#2A9D8F')
    ax_loss.set_yscale('log')
    ax_loss.set_xlabel("Epoch")
    ax_loss.set_title("Training loss")
    ax_loss.legend(fontsize=8)
    ax_loss.grid(True, which='both', alpha=0.3)

    plt.tight_layout()
    plt.show()


for exp in experiments:
    plot_experiment(exp)

## Cross-experiment comparison

In [ ]:
names    = [e['name'] for e in experiments]
chamfers = [e['metrics']['chamfer']   for e in experiments]
energies = [e['metrics']['energy']    for e in experiments]
hinges   = [e['metrics']['hinge_gap'] for e in experiments]
totals   = [e['metrics']['total']     for e in experiments]

# ── Metrics table ─────────────────────────────────────────────────────────────
print(f"{'Experiment':<40} {'Chamfer':>10} {'Energy':>10} {'HingeGap':>10} {'TotalLoss':>12}")
print('-' * 84)
for e in experiments:
    m = e['metrics']
    print(f"{e['name']:<40} {m['chamfer']:>10.4f} {m['energy']:>10.4f} "
          f"{m['hinge_gap']:>10.2f} {m['total']:>12.2f}")

# ── Bar charts ────────────────────────────────────────────────────────────────
short_names = [n.replace('exp', 'E').replace('_', '\n', 1) for n in names]
x = np.arange(len(names))
colors = plt.cm.tab10(np.linspace(0, 1, len(names)))

fig, axes = plt.subplots(1, 3, figsize=(18, 4))
fig.suptitle("Cross-experiment comparison", fontsize=13)

for ax, vals, title, ylabel in [
    (axes[0], chamfers, "Chamfer distance (↓ better)",       "Chamfer (unweighted)"),
    (axes[1], energies, "Physical energy at equilibrium (↓)", "Total energy"),
    (axes[2], hinges,   "Hinge gap (weighted, ↓ better)",     "Hinge gap loss"),
]:
    bars = ax.bar(x, vals, color=colors, edgecolor='black', linewidth=0.5)
    ax.set_xticks(x)
    ax.set_xticklabels(short_names, fontsize=8)
    ax.set_title(title, fontsize=10)
    ax.set_ylabel(ylabel, fontsize=9)
    ax.grid(axis='y', alpha=0.3)
    # Annotate values on bars
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() * 1.02,
                f"{v:.3f}", ha='center', va='bottom', fontsize=7)

plt.tight_layout()
plt.show()

In [ ]:
# ── Stage 2 overlay: all experiments on one target circle ────────────────────
# Useful to visually see how each loading condition deforms the structure
# relative to the same target shape.

from problem.targets import get_target_points

ncols = min(4, len(experiments))
nrows = (len(experiments) + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 5 * nrows))
axes = np.array(axes).flatten()
fig.suptitle("Stage 2 — Physical equilibrium for all experiments", fontsize=13, y=1.01)

for i, exp in enumerate(experiments):
    ax  = axes[i]
    cfg = exp['config']
    res = exp['result']
    tess = exp['tess']
    m    = exp['metrics']

    target_params = {'type': cfg.target.type,
                     'center': cfg.target.center,
                     'radius': cfg.target.radius}

    eq_state = equilibrium_state(res['valid_state'], res['solution'])
    t2_tess  = tess_from_state(eq_state, tess)
    plot_tessellation(
        t2_tess, ax=ax,
        title=f"{exp['name']}\nchamfer={m['chamfer']:.4f}  E={m['energy']:.3f}",
        show_target=True, target_params=target_params,
        show_hinges=True, show_hinge_indices=False,
        show_face_indices=False, show_external_forces=True,
        show_kinematic_blocks=True,
    )

# Hide unused axes
for j in range(len(experiments), len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.show()